# Customer Churn Prediction

End-to-end IBM Telco churn analysis. Run from project root after placing the public source CSV under data/. Model development uses cross-validation on training data and an untouched final test split.

## 1. Import Libraries

In [ ]:
from pathlib import Path
import sys
ROOT = Path.cwd()
if not (ROOT / 'config.py').exists(): ROOT = Path.cwd().parent
sys.path.insert(0, str(ROOT))
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd
from config import FIGURES_DIR, RESULTS_PATH, MODEL_PATH
from src.data_cleaning import load_and_clean_data, categorical_value_summary
from src.preprocessing import engineer_features, make_preprocessor
from src.train import train


## 2. Load Dataset ? 3. Basic Dataset Information

In [ ]:
df = load_and_clean_data()
print('Shape:', df.shape)
display(df.head())
display(df.describe(include='all').T)
print('Categorical unique values:')
for column, values in categorical_value_summary(df).items(): print(f'{column}: {values}')

## 4. Data Cleaning ? 5. Missing Values ? 6. Duplicate Check

In [ ]:
print('Missing values after cleaning:')
display(df.isna().sum().to_frame('missing'))
print('Exact duplicate rows after cleaning:', df.duplicated().sum())
print('TotalCharges is numeric:', pd.api.types.is_numeric_dtype(df['TotalCharges']))

## 7. Target Distribution ? 8. EDA

In [ ]:
FIGURES_DIR.mkdir(parents=True, exist_ok=True)
eda_cats = ['gender','SeniorCitizen','Partner','Dependents','Contract','InternetService','PaymentMethod','TechSupport','OnlineSecurity']
eda_nums = ['tenure','MonthlyCharges','TotalCharges']
fig, axes = plt.subplots(1, 2, figsize=(11,4))
df['Churn'].map({0:'No',1:'Yes'}).value_counts().plot.bar(ax=axes[0], title='Churn Distribution', color=['#4c78a8','#f58518'])
(df['Churn'].value_counts(normalize=True).sort_index()*100).rename(index={0:'No',1:'Yes'}).plot.bar(ax=axes[1], title='Churn Percentage (%)', color=['#4c78a8','#f58518'])
plt.tight_layout(); plt.savefig(FIGURES_DIR/'eda_churn_distribution.png', dpi=160); plt.show()
for column in eda_cats:
    if column not in df: continue
    rates = df.groupby(column, observed=False)['Churn'].mean().sort_values(ascending=False)
    ax = rates.plot.bar(figsize=(8,4), title=f'Churn Rate by {column}', ylabel='Churn rate')
    plt.tight_layout(); plt.savefig(FIGURES_DIR/f'eda_{column.lower()}_vs_churn.png', dpi=160); plt.show()
for column in eda_nums:
    if column not in df: continue
    fig, ax = plt.subplots(figsize=(8,4))
    sns.boxplot(data=df, x='Churn', y=column, ax=ax); ax.set_title(f'{column} by Churn')
    plt.tight_layout(); plt.savefig(FIGURES_DIR/f'eda_{column.lower()}_vs_churn.png', dpi=160); plt.show()
numeric = df.select_dtypes(include='number')
plt.figure(figsize=(10,7)); sns.heatmap(numeric.corr(), annot=True, cmap='coolwarm', center=0, fmt='.2f')
plt.title('Numerical Feature Correlation'); plt.tight_layout(); plt.savefig(FIGURES_DIR/'eda_correlation_heatmap.png', dpi=160); plt.show()

## 9. Feature Engineering

In [ ]:
features = engineer_features(df.drop(columns=['Churn']))
display(features.head())
print('Added:', [c for c in ['tenure_group','service_count','average_monthly_spend'] if c in features])

## 10. Train/Test Split ? 11. Preprocessing ? 12. Imbalance Strategy

In [ ]:
# src.train splits raw rows first, tunes class-weighted candidate pipelines with 5-fold CV,
# then evaluates the selected pipeline once on the untouched test split. Vanilla SMOTE on
# one-hot data is avoided because it can create invalid fractional categorical indicators.
print('See src/train.py for cross-validated model selection and final test evaluation.')

## 13. Logistic Regression ? 14. Random Forest ? 15. XGBoost ? 16. Model Evaluation ? 17. Confusion Matrices ? 18. ROC Curve ? 19. Feature Importance ? 20. Model Comparison

In [ ]:
results, selected_model = train()
display(results)
print('Selected using highest 5-fold training PR-AUC:', selected_model)
print('False negative: actual churn customer predicted as non-churn; this may mean a missed retention opportunity.')
print('Generated figures:', *sorted(p.name for p in FIGURES_DIR.glob('*.png')), sep='\n- ')

## 21. Final Model Saving ? 22. Conclusion

In [ ]:
print('Saved fitted complete pipeline:', MODEL_PATH, MODEL_PATH.exists())
print('Measured results:', RESULTS_PATH, RESULTS_PATH.exists())
print('The held-out metrics describe this dataset split. Select operating thresholds using business costs and outreach capacity.')